In [2]:
import numpy as np
import pandas as pd
# Product Recommendation Dataset
np.random.seed(42)
n_samples = 10000

categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Books', 'Toys', 'Sports', 'Beauty']
subcategories = {
    'Electronics': ['Laptops', 'Phones', 'Headphones', 'Tablets'],
    'Clothing': ['Men', 'Women', 'Kids', 'Accessories'],
    'Home & Kitchen': ['Furniture', 'Cookware', 'Decor', 'Appliances'],
    'Books': ['Fiction', 'Non-fiction', 'Educational', 'Comics'],
    'Toys': ['Action Figures', 'Board Games', 'Educational', 'Outdoor'],
    'Sports': ['Fitness', 'Outdoor', 'Team Sports', 'Equipment'],
    'Beauty': ['Skincare', 'Makeup', 'Haircare', 'Fragrance']
}

data = {
    'user_id': [f'U{str(i).zfill(5)}' for i in range(n_samples)],
    'user_age': np.random.randint(18, 70, n_samples),
    'user_gender': np.random.choice(['M', 'F', 'Other'], n_samples, p=[0.48, 0.48, 0.04]),
    'purchase_history_category': np.random.choice(categories, n_samples),
    'total_spent': np.random.uniform(0, 5000, n_samples).round(2),
    'avg_rating_given': np.random.uniform(1, 5, n_samples).round(1),
    'browsing_time_min': np.random.randint(0, 120, n_samples),
    'items_in_cart': np.random.randint(0, 10, n_samples),
    'wishlist_items': np.random.randint(0, 30, n_samples),
    'season': np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_samples),
    'previous_category': np.random.choice(categories, n_samples),
    'clicked_ads': np.random.randint(0, 20, n_samples)
}

df_recommendation = pd.DataFrame(data)

# Generate target: next_category to recommend
# Simple rule-based: recommend based on age and purchase history
def get_recommended_category(row):
    if row['user_age'] < 25:
        if row['purchase_history_category'] in ['Electronics', 'Toys']:
            return np.random.choice(['Electronics', 'Toys', 'Sports'])
        return 'Electronics'
    elif row['user_age'] < 40:
        if row['purchase_history_category'] in ['Clothing', 'Beauty']:
            return np.random.choice(['Clothing', 'Beauty', 'Home & Kitchen'])
        return 'Home & Kitchen'
    else:
        if row['purchase_history_category'] in ['Books', 'Home & Kitchen']:
            return np.random.choice(['Books', 'Home & Kitchen', 'Sports'])
        return 'Books'

df_recommendation['recommended_category'] = df_recommendation.apply(get_recommended_category, axis=1)

df_recommendation.to_csv('recommendation_dataset.csv', index=False)
print("Recommendation dataset shape:", df_recommendation.shape)

Recommendation dataset shape: (10000, 13)


# Task
Build and evaluate a classification model to predict the 'recommended_category' using the `df_recommendation` DataFrame.

## Prepare Data for Machine Learning

### Subtask:
Convert categorical features in `df_recommendation` into numerical format using one-hot encoding, and separate the features (X) from the target variable (y), which is 'recommended_category'.


**Reasoning**:
The subtask requires converting categorical features into a numerical format using one-hot encoding and separating the features (X) from the target variable (y). I'll start by dropping the 'user_id' column, then identify all categorical columns, apply one-hot encoding to them, and finally split the DataFrame into X and y.



In [3]:
df_processed = df_recommendation.drop('user_id', axis=1)

# Identify categorical columns
categorical_features = df_processed.select_dtypes(include=['object']).columns.tolist()

# Apply one-hot encoding
df_encoded = pd.get_dummies(df_processed, columns=categorical_features, drop_first=False)

# Separate features (X) and target (y)
y = df_encoded.filter(like='recommended_category_')
X = df_encoded.drop(columns=y.columns)

print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")

Shape of X (features): (10000, 28)
Shape of y (target): (10000, 7)


## Split Data into Training and Testing Sets

### Subtask:
Divide the preprocessed dataset into training and testing subsets to properly evaluate the model's performance on unseen data.


**Reasoning**:
To split the data into training and testing sets, I need to import the `train_test_split` function and then apply it to the `X` and `y` DataFrames.



In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (8000, 28)
Shape of X_test: (2000, 28)
Shape of y_train: (8000, 7)
Shape of y_test: (2000, 7)


## Train a Classification Model

### Subtask:
Choose and train a suitable classification algorithm, such as RandomForestClassifier, on the training data to predict the 'recommended_category'.


**Reasoning**:
To train a classification model, I will import `RandomForestClassifier`, instantiate it with `random_state=42`, and then fit the model using the training data `X_train` and `y_train`.



In [5]:
from sklearn.ensemble import RandomForestClassifier

# Instantiate the RandomForestClassifier model
model = RandomForestClassifier(random_state=42)

# Fit the model to the training data
model.fit(X_train, y_train)

print("RandomForestClassifier model trained successfully.")

RandomForestClassifier model trained successfully.


## Evaluate Model Performance

### Subtask:
Assess the trained model's accuracy and other relevant metrics (e.g., precision, recall, F1-score) on the test set.


**Reasoning**:
To evaluate the model's performance, I will first make predictions on the test set, then convert the one-hot encoded true and predicted labels to single-label arrays, and finally calculate and print the accuracy, precision, recall, and F1-score.



In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Make predictions on the test set
y_pred = model.predict(X_test)

# Convert one-hot encoded y_test and y_pred to single-label arrays
y_test_labels = np.argmax(y_test.values, axis=1)
y_pred_labels = np.argmax(y_pred, axis=1)

# Calculate accuracy
accuracy = accuracy_score(y_test_labels, y_pred_labels)

# Calculate precision, recall, and F1-score with 'weighted' average
precision = precision_score(y_test_labels, y_pred_labels, average='weighted', zero_division=0)
recall = recall_score(y_test_labels, y_pred_labels, average='weighted', zero_division=0)
f1 = f1_score(y_test_labels, y_pred_labels, average='weighted', zero_division=0)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"Model Precision (weighted): {precision:.4f}")
print(f"Model Recall (weighted): {recall:.4f}")
print(f"Model F1-score (weighted): {f1:.4f}")

Model Accuracy: 0.7485
Model Precision (weighted): 0.8878
Model Recall (weighted): 0.7485
Model F1-score (weighted): 0.7832


## Final Task

### Subtask:
Summarize the model's accuracy and performance, and discuss the overall effectiveness of the recommendation system.


## Summary:

### Q&A
The classification model achieved an accuracy of 0.7485, a weighted precision of 0.8878, a weighted recall of 0.7485, and a weighted F1-score of 0.7832. Overall, the model demonstrates reasonable effectiveness in predicting the 'recommended_category', with a relatively high precision, indicating that when it predicts a category, it is often correct. However, the recall is lower than precision, suggesting it might miss some actual positive categories.

### Data Analysis Key Findings
*   The dataset was successfully preprocessed by dropping the `user_id` column and converting all categorical features, including the target 'recommended\_category', into a numerical format using one-hot encoding.
*   The data was split into training (80%) and testing (20%) sets, resulting in 8000 samples for training and 2000 samples for testing.
*   A RandomForestClassifier model was trained to predict the 'recommended\_category'.
*   The model's performance on the test set is as follows:
    *   **Accuracy:** 0.7485
    *   **Weighted Precision:** 0.8878
    *   **Weighted Recall:** 0.7485
    *   **Weighted F1-score:** 0.7832

### Insights or Next Steps
*   The model shows good predictive capability, particularly in its precision. To further enhance the recommendation system's effectiveness, it would be beneficial to investigate the specific categories where recall is lower to understand misclassification patterns.
*   Consider exploring other advanced classification models or hyperparameter tuning the current RandomForestClassifier to potentially improve recall without significantly sacrificing precision, which could lead to more comprehensive recommendations.


# Task
Train a GradientBoostingClassifier model on the training data (`X_train`, `y_train`), evaluate its performance on the test set, and summarize its accuracy, precision, recall, and F1-score compared to the previously trained RandomForestClassifier model.

## Train a New Classification Model

### Subtask:
Train a GradientBoostingClassifier model on the training data (`X_train`, `y_train`) to predict the 'recommended_category'.


**Reasoning**:
To train a GradientBoostingClassifier model, I need to import the classifier, convert the one-hot encoded `y_train` to a single-label format, instantiate the model, and then fit it to the training data.



In [7]:
from sklearn.ensemble import GradientBoostingClassifier

# Convert one-hot encoded y_train to single-label array
y_train_labels = np.argmax(y_train.values, axis=1)

# Instantiate the GradientBoostingClassifier model
gbc_model = GradientBoostingClassifier(random_state=42)

# Fit the model to the training data
gbc_model.fit(X_train, y_train_labels)

print("GradientBoostingClassifier model trained successfully.")

GradientBoostingClassifier model trained successfully.


## Evaluate New Model Performance

### Subtask:
Assess the GradientBoostingClassifier model's accuracy and other relevant metrics (e.g., precision, recall, F1-score) on the test set.


**Reasoning**:
To evaluate the GradientBoostingClassifier model's performance, I will make predictions on the test set, convert the true labels to a single-label format, and then compute and print the accuracy, precision, recall, and F1-score.



In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Make predictions on the test set
gbc_y_pred = gbc_model.predict(X_test)

# Convert one-hot encoded y_test to single-label array
y_test_labels_gbc = np.argmax(y_test.values, axis=1)

# Calculate accuracy
gbc_accuracy = accuracy_score(y_test_labels_gbc, gbc_y_pred)

# Calculate precision, recall, and F1-score with 'weighted' average
gbc_precision = precision_score(y_test_labels_gbc, gbc_y_pred, average='weighted', zero_division=0)
gbc_recall = recall_score(y_test_labels_gbc, gbc_y_pred, average='weighted', zero_division=0)
gbc_f1 = f1_score(y_test_labels_gbc, gbc_y_pred, average='weighted', zero_division=0)

print(f"GradientBoostingClassifier Model Accuracy: {gbc_accuracy:.4f}")
print(f"GradientBoostingClassifier Model Precision (weighted): {gbc_precision:.4f}")
print(f"GradientBoostingClassifier Model Recall (weighted): {gbc_recall:.4f}")
print(f"GradientBoostingClassifier Model F1-score (weighted): {gbc_f1:.4f}")

GradientBoostingClassifier Model Accuracy: 0.8100
GradientBoostingClassifier Model Precision (weighted): 0.8032
GradientBoostingClassifier Model Recall (weighted): 0.8100
GradientBoostingClassifier Model F1-score (weighted): 0.8058


## Final Task

### Subtask:
Summarize the new model's accuracy and performance, and discuss the overall effectiveness compared to the previous model.


## Summary:

### Data Analysis Key Findings

*   A GradientBoostingClassifier model was successfully trained to predict the 'recommended\_category'.
*   The GradientBoostingClassifier model achieved the following performance metrics on the test set:
    *   Accuracy: 0.8100
    *   Weighted Precision: 0.8032
    *   Weighted Recall: 0.8100
    *   Weighted F1-score: 0.8058

### Insights or Next Steps

*   To fully address the task, explicitly compare these performance metrics with those of the previously trained RandomForestClassifier model to determine the overall effectiveness of the new model.
*   Consider hyperparameter tuning for the GradientBoostingClassifier to potentially further improve its performance.


# Task
Train an XGBoostClassifier model on the training data (`X_train`, `y_train`) to predict the 'recommended_category'.

## Train an XGBoostClassifier Model

### Subtask:
Train an XGBoostClassifier model on the training data (`X_train`, `y_train`) to predict the 'recommended_category'.


**Reasoning**:
To train an XGBoostClassifier model, I will import `XGBClassifier`, convert the one-hot encoded `y_train` to a single-label array, instantiate the model with a `random_state` for reproducibility, and then fit it to the training data.



In [9]:
from xgboost import XGBClassifier

# Convert one-hot encoded y_train to single-label array
y_train_labels_xgb = np.argmax(y_train.values, axis=1)

# Instantiate the XGBClassifier model
xgb_model = XGBClassifier(random_state=42)

# Fit the model to the training data
xgb_model.fit(X_train, y_train_labels_xgb)

print("XGBoostClassifier model trained successfully.")

XGBoostClassifier model trained successfully.


## Evaluate XGBoostClassifier Performance

### Subtask:
Assess the XGBoostClassifier model's accuracy and other relevant metrics (e.g., precision, recall, F1-score) on the test set.


**Reasoning**:
To evaluate the XGBoostClassifier model's performance, I will make predictions on the test set, convert the true labels to a single-label format, and then compute and print the accuracy, precision, recall, and F1-score.



In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Make predictions on the test set
xgb_y_pred = xgb_model.predict(X_test)

# Convert one-hot encoded y_test to single-label array
y_test_labels_xgb = np.argmax(y_test.values, axis=1)

# Calculate accuracy
xgb_accuracy = accuracy_score(y_test_labels_xgb, xgb_y_pred)

# Calculate precision, recall, and F1-score with 'weighted' average
xgb_precision = precision_score(y_test_labels_xgb, xgb_y_pred, average='weighted', zero_division=0)
xgb_recall = recall_score(y_test_labels_xgb, xgb_y_pred, average='weighted', zero_division=0)
xgb_f1 = f1_score(y_test_labels_xgb, xgb_y_pred, average='weighted', zero_division=0)

print(f"XGBoostClassifier Model Accuracy: {xgb_accuracy:.4f}")
print(f"XGBoostClassifier Model Precision (weighted): {xgb_precision:.4f}")
print(f"XGBoostClassifier Model Recall (weighted): {xgb_recall:.4f}")
print(f"XGBoostClassifier Model F1-score (weighted): {xgb_f1:.4f}")

XGBoostClassifier Model Accuracy: 0.8060
XGBoostClassifier Model Precision (weighted): 0.7944
XGBoostClassifier Model Recall (weighted): 0.8060
XGBoostClassifier Model F1-score (weighted): 0.7986


## Summary:

### Data Analysis Key Findings
*   **RandomForestClassifier Model Performance:**
    *   Accuracy: 0.7485
    *   Weighted Precision: 0.8878
    *   Weighted Recall: 0.7485
    *   Weighted F1-score: 0.7832

*   **GradientBoostingClassifier Model Performance:**
    *   Accuracy: 0.8100
    *   Weighted Precision: 0.8032
    *   Weighted Recall: 0.8100
    *   Weighted F1-score: 0.8058

*   **XGBoostClassifier Model Performance:**
    *   Accuracy: 0.8060
    *   Weighted Precision: 0.7944
    *   Weighted Recall: 0.8060
    *   Weighted F1-score: 0.7986

### Insights or Next Steps
*   The **GradientBoostingClassifier** demonstrated the best overall performance among the three models, achieving the highest accuracy (0.8100), recall (0.8100), and F1-score (0.8058). While its precision (0.8032) was slightly lower than that of the RandomForestClassifier, the balanced performance across all metrics makes it a strong contender for this recommendation system.
*   The **XGBoostClassifier** performed very similarly to the GradientBoostingClassifier, with slightly lower but comparable metrics (Accuracy: 0.8060, Precision: 0.7944, Recall: 0.8060, F1-score: 0.7986).
*   The **RandomForestClassifier**, while having the highest precision (0.8878), exhibited a lower accuracy, recall, and F1-score compared to both Gradient Boosting models, suggesting it might be more conservative in its predictions but misses more actual positive cases.
*   For future improvements, hyperparameter tuning for both GradientBoostingClassifier and XGBoostClassifier could lead to even better performance. Additionally, exploring ensemble methods that combine the strengths of these models could be beneficial. Further analysis into specific misclassifications could help refine feature engineering or model selection.

## Final Task

### Subtask:
Summarize the XGBoostClassifier model's performance and compare it with the RandomForestClassifier and GradientBoostingClassifier models to determine the best-performing model.


## Summary:

### Q&A
The best-performing model among the three evaluated is the GradientBoostingClassifier, which achieved the highest accuracy, recall, and F1-score.

### Data Analysis Key Findings
*   The **GradientBoostingClassifier** exhibited the best overall performance with an accuracy of 0.8100, weighted precision of 0.8032, weighted recall of 0.8100, and a weighted F1-score of 0.8058.
*   The **XGBoostClassifier** performed very similarly to the GradientBoostingClassifier, achieving an accuracy of 0.8060, weighted precision of 0.7944, weighted recall of 0.8060, and a weighted F1-score of 0.7986.
*   The **RandomForestClassifier** had the highest weighted precision at 0.8878, but it showed lower performance in other metrics, with an accuracy of 0.7485, weighted recall of 0.7485, and a weighted F1-score of 0.7832.

### Insights or Next Steps
*   The GradientBoostingClassifier demonstrated the most balanced and highest performance across key metrics, making it a strong candidate for the recommendation system.
*   Future efforts should focus on hyperparameter tuning for both GradientBoostingClassifier and XGBoostClassifier to potentially enhance their performance further, and consider exploring ensemble methods.
